# ROPG-KD Data Generation
Generates `data/ropg_kd/{train,val}.jsonl` — scored (query, persona, top-K chunks) triples used to train the ROPG-KD retriever.
The production generator is `src/data/gen_ropg_data.py`; regenerate this standalone notebook with `notebooks/build_gen_ropg_data.py` after changing it.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 × 1 is enough).
2. Enable internet access.
3. Add Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL`.
4. Attach the `simurgh-data` dataset (contains `chunks/corpus.jsonl`, `questions/`, `splits/`).

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in the Config cell below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `chunks/corpus.jsonl`, `questions/`, `splits/`.
3. Add secrets via Colab Secrets (left sidebar → key icon): `OPENAI_API_KEY`, `OPENAI_BASE_URL`.
4. Enable GPU accelerator (T4 × 1 is enough).
5. Output is saved to `MyDrive/simurgh-data/ropg_kd/`.

**Format migration (current version 4).** Current scored rows use format version 4. Replacing the judge changes
all teacher scores, and the complete-query and max-sequence-length changes require regenerating all scored
artifacts: old stem-only, format-version-2, and format-version-3 `{train,val}.jsonl` rows and derived files
are incompatible, and `derive-only` cannot repair them. Use a fresh output directory and do not mix old and
new rows. Successful
rows append/resume on reruns; exhausted failed groups remain retryable.

**Re-deriving triplets without re-judging.** The last cell turns the scored
`{train,val}.jsonl` into triplets/pairs, and it makes no API calls. To change
`triplets.max_negatives` or turn the label filters on, edit `CFG` and re-run **only that
cell** — the expensive judge pass above stays untouched. Each run writes
`{split}_triplets_meta.json` recording the settings and the retained/total group counts.



In [ ]:
!pip install -q sentence-transformers==5.6.0 openai pyyaml numpy tqdm

## Generation helpers

In [ ]:
import json
import logging
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path


logging.basicConfig(
    force=True,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

QUESTION_OUTPUT_FORMAT_VERSION = 4
_SCORE_RE = re.compile(r"(?:0(?:\.\d+)?|1(?:\.0+)?)")

JUDGE_SYSTEM = "You are an expert Persian language tutor evaluating study materials."


@dataclass(frozen=True)
class QuestionContext:
    query: str
    answer: object | None = None
    explanation: str | None = None


class JudgeScoringError(RuntimeError):
    pass


def load_corpus(path: Path):
    chunk_ids, texts = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        chunk_ids.append(rec["chunk_id"])
        texts.append(rec["text"])
    if not texts:
        raise ValueError(f"Corpus file {path} is empty")
    return chunk_ids, texts


def load_split_qids(path: Path):
    entries = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if ":" not in line:
            logger.warning("Skipping malformed split line: %r", line)
            continue
        exam_stem, qid = line.split(":", 1)
        entries.append((exam_stem, qid, line))
    return entries


def _render_question_value(value) -> str:
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def _render_numbered_section(label: str, values) -> str | None:
    if values is None:
        return None
    entries = values if isinstance(values, list) else [values]
    if not entries:
        return None
    rendered = "\n".join(
        f"{index}. {_render_question_value(value)}"
        for index, value in enumerate(entries, start=1)
    )
    return f"{label}:\n{rendered}"


def load_question(exam_stem: str, qid: str, questions_dir: Path) -> QuestionContext:
    question_path = questions_dir / f"{exam_stem}.json"
    if not question_path.exists():
        raise FileNotFoundError(f"Question file not found: {question_path}")
    data = json.loads(question_path.read_text(encoding="utf-8"))
    for question in data.get("questions", []):
        if question["id"] != qid:
            continue
        sections = []
        group_id = question.get("group_id")
        if group_id is not None:
            passages = data.get("passages", data.get("passage", []))
            matching_passage = None
            if isinstance(passages, dict):
                if passages.get("id") == group_id or passages.get("group_id") == group_id:
                    matching_passage = passages
                else:
                    matching_passage = passages.get(group_id)
                    if matching_passage is None:
                        matching_passage = passages.get(str(group_id))
            elif isinstance(passages, list):
                for passage in passages:
                    if not isinstance(passage, dict):
                        continue
                    if passage.get("id") == group_id or passage.get("group_id") == group_id:
                        matching_passage = passage
                        break
            if matching_passage is None:
                raise KeyError(
                    f"Question {qid!r} in {question_path} references group_id={group_id!r}, "
                    "but no matching top-level passage exists"
                )
            if isinstance(matching_passage, dict):
                passage_text = matching_passage.get("text", matching_passage.get("passage"))
            else:
                passage_text = matching_passage
            if passage_text is None:
                raise KeyError(
                    f"Question {qid!r} in {question_path} references group_id={group_id!r}, "
                    "but the matching top-level passage has no text"
                )
            sections.append(f"Passage:\n{_render_question_value(passage_text)}")

        sections.append(f"Question:\n{_render_question_value(question['stem'])}")
        options_section = _render_numbered_section("Options", question.get("options"))
        if options_section is not None:
            sections.append(options_section)
        pairs = question.get("pairs")
        if isinstance(pairs, dict):
            for side in ("left", "right"):
                pair_section = _render_numbered_section(
                    f"Pairs ({side})", pairs.get(side)
                )
                if pair_section is not None:
                    sections.append(pair_section)
        elif pairs is not None:
            pair_section = _render_numbered_section("Pairs", pairs)
            if pair_section is not None:
                sections.append(pair_section)
        items_section = _render_numbered_section("Items", question.get("items"))
        if items_section is not None:
            sections.append(items_section)
        return QuestionContext(
            query="\n\n".join(sections),
            answer=question.get("answer"),
            explanation=question.get("explanation"),
        )
    raise KeyError(f"Question {qid!r} not found in {question_path}")


def retrieve_top_k(query_vec, chunk_matrix, top_k: int, chunk_ids, chunk_texts):
    sims = query_vec.squeeze() @ chunk_matrix.T
    top_idx = np.argsort(sims)[-top_k:][::-1]
    return [chunk_ids[i] for i in top_idx], [chunk_texts[i] for i in top_idx]


def parse_float_score(response: str) -> float:
    if not isinstance(response, str):
        raise ValueError(f"Judge response is not a score string: {response!r}")
    text = response.strip()
    if _SCORE_RE.fullmatch(text) is None:
        raise ValueError(f"Judge response is not a single score in [0, 1]: {response!r}")
    score = float(text)
    if not math.isfinite(score) or not 0.0 <= score <= 1.0:
        raise ValueError(f"Judge response is not a finite score in [0, 1]: {response!r}")
    return score


def build_judge_messages(query, persona_rendered: str, chunk_text: str, *, answer=None, explanation=None):
    # Gold fields are passed separately to the judge and never become part of query.
    reference_sections = []
    if answer is not None:
        rendered_answer = answer if isinstance(answer, str) else _render_question_value(answer)
        reference_sections.append(f"Gold answer/reference:\n{rendered_answer}")
    if explanation is not None:
        reference_sections.append(f"Gold explanation/rubric:\n{explanation}")
    reference_context = ""
    if reference_sections:
        reference_context = (
            "Reference answer and rubric (judge context only; not part of the retrieval query):\n"
            + "\n\n".join(reference_sections)
            + "\n\n"
        )
    user = (
        "A student with the following profile is trying to answer an exam question:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Exam question: {query}\n\n"
        f"{reference_context}"
        f"Candidate study passage:\n{chunk_text}\n\n"
        "Rate 0.0–1.0 how useful this passage is for helping this specific student answer "
        "the question. Consider:\n"
        "  - Does the depth match the student's comprehension level?\n"
        "  - Does it provide what this student needs (simple paraphrase vs. deep analysis)?\n"
        "  - Is the style appropriate (hand-holding vs. terse treatment)?\n\n"
        "Respond with a single decimal number only, e.g. 0.73"
    )
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]


def _validate_retry_settings(max_attempts, initial_backoff_seconds, backoff_multiplier, max_backoff_seconds):
    if isinstance(max_attempts, bool) or not isinstance(max_attempts, int) or max_attempts < 1:
        raise ValueError("max_attempts must be an integer >= 1")
    values = (initial_backoff_seconds, backoff_multiplier, max_backoff_seconds)
    if any(
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(float(value))
        for value in values
    ):
        raise ValueError("retry delays and multiplier must be finite numbers")
    if initial_backoff_seconds < 0 or max_backoff_seconds < 0:
        raise ValueError("retry delays must be >= 0")
    if backoff_multiplier < 1:
        raise ValueError("backoff_multiplier must be >= 1")
    if max_backoff_seconds < initial_backoff_seconds:
        raise ValueError("max_backoff_seconds must be >= initial_backoff_seconds")


def score_one_chunk(
    judge,
    query,
    persona_rendered: str,
    chunk_id,
    chunk_text: str,
    *,
    max_attempts: int = 3,
    initial_backoff_seconds: float = 1.0,
    backoff_multiplier: float = 2.0,
    max_backoff_seconds: float = 8.0,
    answer=None,
    explanation=None,
):
    _validate_retry_settings(max_attempts, initial_backoff_seconds, backoff_multiplier, max_backoff_seconds)
    for attempt in range(1, max_attempts + 1):
        try:
            response = judge.chat(
                build_judge_messages(
                    query, persona_rendered, chunk_text, answer=answer, explanation=explanation
                )
            )
            score = parse_float_score(response)
            return {"chunk_id": chunk_id, "text": chunk_text, "teacher_score": score}
        except Exception as exc:
            logger.warning(
                "Judge chunk %s attempt %d/%d failed: %s", chunk_id, attempt, max_attempts, exc
            )
            if attempt == max_attempts:
                raise JudgeScoringError(
                    f"Judge scoring exhausted for chunk {chunk_id!r} after {max_attempts} attempts"
                ) from exc
            delay = min(
                float(max_backoff_seconds),
                float(initial_backoff_seconds) * (float(backoff_multiplier) ** (attempt - 1)),
            )
            if delay:
                time.sleep(delay)
    raise AssertionError("unreachable")


def score_chunks(
    judge,
    query,
    persona_rendered: str,
    chunk_ids,
    chunk_texts,
    max_workers: int,
    max_attempts: int = 3,
    initial_backoff_seconds: float = 1.0,
    backoff_multiplier: float = 2.0,
    max_backoff_seconds: float = 8.0,
    *,
    answer=None,
    explanation=None,
):
    if len(chunk_ids) != len(chunk_texts):
        raise ValueError("chunk_ids and chunk_texts must have equal lengths")
    _validate_retry_settings(max_attempts, initial_backoff_seconds, backoff_multiplier, max_backoff_seconds)
    results = [None] * len(chunk_ids)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                score_one_chunk,
                judge,
                query,
                persona_rendered,
                chunk_id,
                chunk_text,
                max_attempts=max_attempts,
                initial_backoff_seconds=initial_backoff_seconds,
                backoff_multiplier=backoff_multiplier,
                max_backoff_seconds=max_backoff_seconds,
                answer=answer,
                explanation=explanation,
            ): index
            for index, (chunk_id, chunk_text) in enumerate(zip(chunk_ids, chunk_texts))
        }
        try:
            for future in as_completed(futures):
                results[futures[future]] = future.result()
        except Exception as exc:
            for future in futures:
                future.cancel()
            if isinstance(exc, JudgeScoringError):
                raise
            raise JudgeScoringError("Judge scoring failed for chunk group") from exc
    return results


## Config

In [ ]:
import os

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Secrets ───────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    from kaggle_secrets import UserSecretsClient

    _s = UserSecretsClient()
    OPENAI_API_KEY = _s.get_secret("OPENAI_API_KEY")
    OPENAI_BASE_URL = _s.get_secret("OPENAI_BASE_URL")
elif RUNTIME == "colab":
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    OPENAI_BASE_URL = userdata.get("OPENAI_BASE_URL")
else:  # local
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    OUTPUT_DIR = "/kaggle/working/data/ropg_kd"
elif RUNTIME == "colab":
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/ropg_kd"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/ropg_kd"

# ── Inline config (mirrors configs/datagen_ropg.yaml) ────────────────────────
CFG = {
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "device": "cuda",  # Kaggle/Colab GPU
        "batch_size": 8,
        "fp16": True,
        "max_seq_length": 2048,
    },
    "retriever": {"top_k": 20},
    "judge": {
        "model": "gpt-5.6-luna",
        "reasoning_effort": "none",
        "temperature": 1.0,
        "max_completion_tokens": 16,
        "max_workers": 8,
        "max_attempts": 3,
        "initial_backoff_seconds": 1.0,
        "backoff_multiplier": 2.0,
        "max_backoff_seconds": 8.0,  # retry API/malformed responses; skip group atomically after exhaustion
    },
    "data": {
        "chunks": f"{DATA_ROOT}/chunks/corpus.jsonl",
        "questions_dir": f"{DATA_ROOT}/questions",
        "splits_dir": f"{DATA_ROOT}/splits",
        "output_dir": OUTPUT_DIR,
    },
    # Derivation of hard-negative triplets/pairs from the scored data. Needs no judge
    # calls: it re-reads the {train,val}.jsonl this notebook already wrote, so changing
    # anything here costs seconds and only the last cell has to be re-run.
    "triplets": {
        # 4 -> 8 costs almost no signal (mean positive-negative gap 0.711 -> 0.686) and
        # doubles the terms in the MNRL softmax. Must be >= training's max_negatives.
        "max_negatives": 8,
        # The judge's per-label reliability is ~0.17, so a large share of groups carry
        # supervision that is noise rather than signal. Default off: the unfiltered arm
        # must stay reproducible from config alone.
        "filters": {
            "enabled": False,
            # rank-1 and rank-2 within this => which one is "the positive" is a coin
            # flip. 34% of train groups fail this at 0.05.
            "min_positive_margin": 0.08,
            # even the best candidate this weak => the judge found nothing useful and
            # the positive is noise by construction. 10% of train groups fail at 0.5.
            "min_positive_score": 0.4,
            # drop an individual negative sitting too close to the positive to be safe.
            "min_negative_margin": 0.3,
        },
    },
    "seed": 42,
}

## Core classes

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


class Qwen3Embedder:
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-Embedding-0.6B",
        device: str = "cuda",
        batch_size: int = 4,
        fp16: bool = True,
        max_seq_length: int | None = None,
    ) -> None:
        model_kwargs = {"torch_dtype": torch.float16} if fp16 else {}
        self.model = SentenceTransformer(
            model_name, device=device, trust_remote_code=True, model_kwargs=model_kwargs
        )
        if max_seq_length is not None:
            self.model.max_seq_length = max_seq_length
        self.batch_size = batch_size
        self.dim: int = self.model.get_embedding_dimension()

    def encode(self, texts: list) -> np.ndarray:
        vecs = self.model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        return np.array(vecs, dtype=np.float32)

    def encode_query(self, texts: list, instruction: str = "") -> np.ndarray:
        if instruction:
            prompt = f"Instruct: {instruction}\nQuery: "
            vecs = self.model.encode(
                texts,
                prompt=prompt,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        else:
            vecs = self.model.encode(
                texts,
                batch_size=self.batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        return np.array(vecs, dtype=np.float32)

In [ ]:
import openai


class OpenAICompatClient:
    def __init__(self, base_url, api_key, model, temperature=0.0, max_tokens=16, reasoning_effort=None):
        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.reasoning_effort = reasoning_effort

    def chat(self, messages: list) -> str:
        request = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_completion_tokens": self.max_tokens,
        }
        if self.reasoning_effort is not None:
            request["reasoning_effort"] = self.reasoning_effort

        resp = self.client.chat.completions.create(**request)
        return resp.choices[0].message.content or ""

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

## Run pipeline

In [ ]:
np.random.seed(CFG["seed"])

corpus_path = Path(CFG["data"]["chunks"])
questions_dir = Path(CFG["data"]["questions_dir"])
splits_dir = Path(CFG["data"]["splits_dir"])
output_dir = Path(CFG["data"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

logger.info("Loading corpus from %s", corpus_path)
chunk_ids, chunk_texts = load_corpus(corpus_path)
logger.info("%d chunks loaded", len(chunk_ids))

In [ ]:
logger.info("Encoding corpus with %s on %s …", CFG["embedder"]["model"], CFG["embedder"]["device"])
embedder = Qwen3Embedder(
    model_name=CFG["embedder"]["model"],
    device=CFG["embedder"]["device"],
    batch_size=CFG["embedder"]["batch_size"],
    fp16=CFG["embedder"]["fp16"],
    max_seq_length=CFG["embedder"]["max_seq_length"],
)
chunk_matrix = embedder.encode(chunk_texts)
logger.info("Corpus matrix shape: %s", chunk_matrix.shape)

In [ ]:
from tqdm.auto import tqdm

judge_cfg = CFG["judge"]
max_attempts = judge_cfg["max_attempts"]
initial_backoff_seconds = judge_cfg["initial_backoff_seconds"]
backoff_multiplier = judge_cfg["backoff_multiplier"]
max_backoff_seconds = judge_cfg["max_backoff_seconds"]
_validate_retry_settings(
    max_attempts,
    initial_backoff_seconds,
    backoff_multiplier,
    max_backoff_seconds,
)

judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=judge_cfg["model"],
    temperature=judge_cfg["temperature"],
    max_tokens=judge_cfg["max_completion_tokens"],
    reasoning_effort=judge_cfg.get("reasoning_effort"),
)

top_k = CFG["retriever"]["top_k"]
train_profiles = train_personas()

for split in ("train", "val"):
    split_path = splits_dir / f"{split}_qids.txt"
    if not split_path.exists():
        logger.warning("Split file not found: %s — skipping", split_path)
        continue

    output_path = output_dir / f"{split}.jsonl"
    seen = set()
    if output_path.exists():
        for line_no, raw in enumerate(output_path.read_text(encoding="utf-8").splitlines(), 1):
            if not raw.strip():
                continue
            try:
                rec = json.loads(raw)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Malformed row in {output_path}:{line_no}") from exc
            if not isinstance(rec, dict):
                raise ValueError(f"Non-object row in {output_path}:{line_no}")
            if rec.get("format_version") != QUESTION_OUTPUT_FORMAT_VERSION:
                raise ValueError(f"Incompatible format_version in {output_path}:{line_no}")
            if "query" not in rec or "persona_id" not in rec:
                raise ValueError(f"Missing query/persona_id in {output_path}:{line_no}")
            seen.add((rec["query"], rec["persona_id"]))

    entries = load_split_qids(split_path)
    logger.info("Processing %s split (%d questions) → %s", split, len(entries), output_path)

    # Pre-load complete questions; retrieval receives only question.query.
    valid_entries = []
    for exam_stem, qid, raw_line in entries:
        try:
            question = load_question(exam_stem, qid, questions_dir)
            valid_entries.append((exam_stem, qid, raw_line, question))
        except (FileNotFoundError, KeyError) as exc:
            logger.warning("Skipping %s: %s", raw_line, exc)

    failed_groups = 0
    with output_path.open("a", encoding="utf-8") as fh:
        for persona in tqdm(train_profiles, desc=f"{split} personas", unit="p"):
            persona_rendered = render_profile(persona.id)
            pending = [
                entry
                for entry in valid_entries
                if (entry[3].query, persona.id) not in seen
            ]
            if not pending:
                continue
            query_vecs = embedder.encode_query(
                [entry[3].query for entry in pending], instruction=persona_rendered
            )
            for i, (_, _, raw_line, question) in enumerate(pending):
                query_vec = query_vecs[i : i + 1]
                top_ids, top_texts = retrieve_top_k(
                    query_vec, chunk_matrix, top_k, chunk_ids, chunk_texts
                )
                try:
                    docs = score_chunks(
                        judge,
                        question.query,
                        persona_rendered,
                        top_ids,
                        top_texts,
                        judge_cfg["max_workers"],
                        max_attempts=max_attempts,
                        initial_backoff_seconds=initial_backoff_seconds,
                        backoff_multiplier=backoff_multiplier,
                        max_backoff_seconds=max_backoff_seconds,
                        answer=question.answer,
                        explanation=question.explanation,
                    )
                except JudgeScoringError as exc:
                    failed_groups += 1
                    logger.error(
                        "Failed group split=%s question=%s persona=%s: %s",
                        split,
                        raw_line,
                        persona.id,
                        exc,
                    )
                    continue
                rec = {
                    "format_version": QUESTION_OUTPUT_FORMAT_VERSION,
                    "question_ref": raw_line,
                    "query": question.query,
                    "persona_id": persona.id,
                    "docs": docs,
                }
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fh.flush()
                seen.add((question.query, persona.id))

    logger.info("%s split failed groups: %d", split, failed_groups)
    logger.info("Wrote/resumed %s", output_path)

logger.info("All splits complete. Output in %s", output_dir)

In [ ]:
# ── Derive hard-negative triplets and pairs from the scored data ──────────────
# VERBATIM copy of derive_triplets from src/data/gen_ropg_data.py. This cell used to
# hold a hand-written re-implementation, which had already drifted from the source it
# was meant to mirror. Do not edit it here: change the source module and re-copy.
#
# Nothing below needs the LLM judge — it re-reads the {train,val}.jsonl written above,
# so re-running only this cell is enough to change max_negatives or the filters.
import json
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

def derive_triplets(
    scored_path: Path,
    output_dir: Path,
    max_negatives: int = 4,
    filters: dict | None = None,
) -> None:
    """Derive hard-negative triplets and pairs from a scored JSONL file.

    Writes two files alongside the scored file:
      {stem}_triplets.jsonl — one line per group: {query, persona_id, positive, negatives:[...]}
      {stem}_pairs.jsonl    — one line per (pos, neg) pair: {query, persona_id, positive, negative}

    Negatives are written **hardest-first** (descending teacher_score), which matters
    because ``TripletDataset`` truncates with ``negatives[:max_negatives]``: a file
    holding more negatives than training requests must hand over the hardest ones,
    not the easiest.

    *filters* drops groups whose supervision is noise rather than signal — an arbitrary
    positive, or a group where the judge found nothing useful at all. It defaults to
    disabled, so the unfiltered arm stays reproducible from config alone.

    Format validation stays in this function so normal generation, ``--derive-only``, and
    the generated Kaggle notebook all reject stale scored data before replacing derived files.
    """
    stem = scored_path.stem  # e.g. "train" or "val"
    triplet_path = output_dir / f"{stem}_triplets.jsonl"
    pairs_path = output_dir / f"{stem}_pairs.jsonl"

    # Validate the complete input before opening derived files in truncate mode. In
    # particular, stem-only data predating complete question rendering must never be
    # made to look current by running --derive-only.
    scored_records: list[dict] = []
    for line_number, raw in enumerate(
        scored_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        if not raw.strip():
            continue
        try:
            rec = json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Malformed JSON in scored data {scored_path} at line {line_number}"
            ) from exc
        if not isinstance(rec, dict):
            raise ValueError(
                f"Scored data {scored_path} line {line_number} is not a JSON object"
            )
        if rec.get("format_version") != QUESTION_OUTPUT_FORMAT_VERSION:
            raise ValueError(
                f"Scored data {scored_path} line {line_number} has format version "
                f"{rec.get('format_version')!r}, expected {QUESTION_OUTPUT_FORMAT_VERSION}; "
                "regenerate scored data instead of using --derive-only"
            )
        scored_records.append(rec)

    filters = filters or {}
    enabled = bool(filters.get("enabled", False))
    # Thresholds are read once here so the resolved values land in the sidecar even
    # when filtering is off — that is what makes a derived file traceable to a config.
    min_positive_margin = float(filters.get("min_positive_margin", 0.08))
    min_positive_score = float(filters.get("min_positive_score", 0.4))
    min_negative_margin = float(filters.get("min_negative_margin", 0.3))

    n_groups = 0
    n_triplets = 0
    n_pairs = 0
    n_dropped = {"too_few_docs": 0, "positive_margin": 0, "positive_score": 0, "no_negatives": 0}

    with (
        triplet_path.open("w", encoding="utf-8") as tf,
        pairs_path.open("w", encoding="utf-8") as pf,
    ):
        for rec in scored_records:
            docs = rec.get("docs", [])
            n_groups += 1
            if len(docs) < 2:
                n_dropped["too_few_docs"] += 1
                continue

            sorted_docs = sorted(docs, key=lambda d: d["teacher_score"], reverse=True)
            pos_score = sorted_docs[0]["teacher_score"]

            if enabled:
                # A positive that ties with rank-2 is arbitrary: the two scores differ by
                # less than the judge's noise, so which one becomes "the positive" is a coin
                # flip. Measured at 34% of train groups for a 0.05 margin.
                if pos_score - sorted_docs[1]["teacher_score"] < min_positive_margin:
                    n_dropped["positive_margin"] += 1
                    continue
                # If even the best candidate is weak, the judge found nothing useful in the
                # retrieved pool and the positive is noise by construction. 10% of train
                # groups score below 0.5 at rank 1.
                if pos_score < min_positive_score:
                    n_dropped["positive_score"] += 1
                    continue

            positive = sorted_docs[0]["text"]
            # Slice off rank-1 before taking the tail: with max_negatives >= len(docs) the
            # window would otherwise reach back far enough to include the positive itself
            # and emit it as its own negative. Real groups hold 20 docs so this does not
            # bite at max_negatives=8, but it is one config bump away from doing so.
            #
            # The tail stays in descending score order, which is already hardest-first:
            # index 0 is the highest-scoring doc the judge still ranked out of contention,
            # so TripletDataset's ``negatives[:max_negatives]`` head slice keeps the hardest
            # ones when a file holds more negatives than training asks for.
            tail = sorted_docs[1:][-max_negatives:]
            if enabled:
                tail = [d for d in tail if pos_score - d["teacher_score"] >= min_negative_margin]
            negatives = [d["text"] for d in tail]
            if not negatives:
                n_dropped["no_negatives"] += 1
                continue

            base = {"query": rec["query"], "persona_id": rec.get("persona_id", "")}

            tf.write(
                json.dumps(
                    {**base, "positive": positive, "negatives": negatives}, ensure_ascii=False
                )
                + "\n"
            )
            n_triplets += 1

            for neg in negatives:
                pf.write(
                    json.dumps({**base, "positive": positive, "negative": neg}, ensure_ascii=False)
                    + "\n"
                )
                n_pairs += 1

    logger.info(
        "Derived %d triplets and %d pairs from %s (%d/%d groups retained) → %s, %s",
        n_triplets,
        n_pairs,
        scored_path.name,
        n_triplets,
        n_groups,
        triplet_path.name,
        pairs_path.name,
    )
    if enabled:
        logger.info("  dropped: %s", ", ".join(f"{k}={v}" for k, v in n_dropped.items() if v))

    # Sidecar: a derived file is otherwise indistinguishable from one built under
    # different thresholds, and the retained counts go straight into the results table.
    meta_path = output_dir / f"{stem}_triplets_meta.json"
    meta_path.write_text(
        json.dumps(
            {
                "source": scored_path.name,
                "max_negatives": max_negatives,
                "filters": {
                    "enabled": enabled,
                    "min_positive_margin": min_positive_margin,
                    "min_positive_score": min_positive_score,
                    "min_negative_margin": min_negative_margin,
                },
                "groups_total": n_groups,
                "groups_retained": n_triplets,
                "pairs": n_pairs,
                "dropped": n_dropped,
            },
            indent=2,
        ),
        encoding="utf-8",
    )


_tri = CFG.get("triplets", {})
for split in ("train", "val"):
    scored_path = Path(OUTPUT_DIR) / f"{split}.jsonl"
    if scored_path.exists():
        derive_triplets(
            scored_path,
            Path(OUTPUT_DIR),
            max_negatives=_tri.get("max_negatives", 4),
            filters=_tri.get("filters"),
        )
    else:
        print(f"  {scored_path} not found — skipping")

print("Triplet derivation complete.")
